# Experiment 3: Classification Model Practice

This experiment builds classification models for employee attrition prediction using the preprocessed data from Experiment 2. The four models are:

1. Information Gain Decision Tree
2. Gini Index Decision Tree
3. Naive Bayes
4. Support Vector Machine

The validation set is split from `X_train_preprocessed.csv` and `y_train_preprocessed.csv`. `X_test_preprocessed.csv` has no true label, so it is only used for final prediction.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

NOTEBOOK_CWD = Path.cwd().resolve()
if (NOTEBOOK_CWD / "experiment3_classification_models.py").exists():
    EXP3_DIR = NOTEBOOK_CWD
elif (NOTEBOOK_CWD / "实验三" / "experiment3_classification_models.py").exists():
    EXP3_DIR = NOTEBOOK_CWD / "实验三"
else:
    EXP3_DIR = Path(__file__).resolve().parent if "__file__" in globals() else NOTEBOOK_CWD

if str(EXP3_DIR) not in sys.path:
    sys.path.insert(0, str(EXP3_DIR))

from experiment3_classification_models import (
    MODEL_CONFIGS,
    align_and_clean_features,
    define_models,
    encode_labels,
    ensure_output_dirs,
    extract_label_series,
    load_data,
    make_analysis_text,
    plot_all_confusion_matrices,
    plot_error_analysis,
    plot_feature_correlation_heatmap,
    plot_model_metrics_comparison,
    plot_prediction_probability_distribution,
    plot_pr_curves,
    plot_roc_curves,
    plot_runtime_comparison,
    plot_target_distribution,
    plot_tree_feature_importance,
    plot_tree_structures,
    predict_test_set,
    print_basic_data_checks,
    resolve_paths,
    save_evaluation_outputs,
    split_training_validation,
    train_and_evaluate_models,
)

paths = resolve_paths(EXP3_DIR)
ensure_output_dirs(paths)

PROJECT_ROOT = paths["project_root"]
EXP2_DIR = paths["exp2_dir"]
RESULTS_DIR = paths["results_dir"]
FIGURES_DIR = paths["figures_dir"]

print(f"Experiment 3 directory: {EXP3_DIR}")
print(f"Experiment 2 data directory: {EXP2_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Figures directory: {FIGURES_DIR}")

In [ ]:
x_train_raw, x_test_raw, y_train_raw = load_data(paths)

print("\nX_train preview:")
display(x_train_raw.head())

print("\ny_train preview:")
display(y_train_raw.head())

print("\nX_test preview:")
display(x_test_raw.head())

In [ ]:
print_basic_data_checks(x_train_raw, x_test_raw, y_train_raw)

y_series = extract_label_series(y_train_raw)
y, label_metadata = encode_labels(y_series)
x_train_full, x_test_full = align_and_clean_features(x_train_raw, x_test_raw)

print("\nLabel metadata:")
display(label_metadata)
print("\nCleaned feature preview:")
display(x_train_full.head())

In [ ]:
target_distribution_path = plot_target_distribution(y, paths)
print(f"Saved: {target_distribution_path}")
display(Image(filename=str(target_distribution_path)))

In [ ]:
correlation_heatmap_path = plot_feature_correlation_heatmap(x_train_full, paths)
print(f"Saved: {correlation_heatmap_path}")
display(Image(filename=str(correlation_heatmap_path)))

In [ ]:
x_train, x_val, y_train, y_val = split_training_validation(x_train_full, y)

print("\nTraining feature shape:", x_train.shape)
print("Validation feature shape:", x_val.shape)

In [ ]:
models = define_models()

for model_key, model in models.items():
    print(f"{MODEL_CONFIGS[model_key]['display_name']}: {model}")

In [ ]:
outputs = train_and_evaluate_models(models, x_train, x_val, y_train, y_val)
ranking_df = save_evaluation_outputs(outputs, paths)

print("Saved result files:")
for filename in [
    "model_evaluation_results.csv",
    "classification_report_all_models.txt",
    "validation_predictions_all_models.csv",
    "model_runtime_results.csv",
    "model_ranking_summary.csv",
]:
    print("-", RESULTS_DIR / filename)

display(outputs["evaluation"])
display(ranking_df)

In [ ]:
model_evaluation_results = outputs["evaluation"].copy()

print("Sorted by accuracy:")
display(model_evaluation_results.sort_values("accuracy", ascending=False))

print("Sorted by recall:")
display(model_evaluation_results.sort_values("recall", ascending=False))

print("Sorted by F1-score:")
display(model_evaluation_results.sort_values("f1_score", ascending=False))

print("Sorted by ROC-AUC:")
display(model_evaluation_results.sort_values("roc_auc", ascending=False))

best_overall = model_evaluation_results.sort_values(
    ["f1_score", "recall", "pr_auc"], ascending=False
).iloc[0]
best_recall = model_evaluation_results.sort_values("recall", ascending=False).iloc[0]

print(f"Overall best by F1/Recall/PR-AUC: {best_overall['model']}")
print(f"Highest recall model: {best_recall['model']}")

In [ ]:
confusion_matrix_paths = plot_all_confusion_matrices(outputs, paths)
for figure_path in confusion_matrix_paths:
    print(f"Saved: {figure_path}")
    display(Image(filename=str(figure_path)))

In [ ]:
metrics_comparison_path = plot_model_metrics_comparison(outputs["evaluation"], paths)
print(f"Saved: {metrics_comparison_path}")
display(Image(filename=str(metrics_comparison_path)))

In [ ]:
roc_curve_path = plot_roc_curves(outputs, paths)
pr_curve_path = plot_pr_curves(outputs, paths)

print(f"Saved: {roc_curve_path}")
display(Image(filename=str(roc_curve_path)))

print(f"Saved: {pr_curve_path}")
display(Image(filename=str(pr_curve_path)))

In [ ]:
tree_structure_paths = plot_tree_structures(
    outputs["models"],
    list(x_train_full.columns),
    paths,
    max_depth=3,
)
for figure_path in tree_structure_paths:
    print(f"Saved: {figure_path}")
    display(Image(filename=str(figure_path)))

In [ ]:
feature_importance_paths = plot_tree_feature_importance(
    outputs["models"],
    list(x_train_full.columns),
    paths,
)
for figure_path in feature_importance_paths:
    print(f"Saved: {figure_path}")
    display(Image(filename=str(figure_path)))

In [ ]:
probability_distribution_path = plot_prediction_probability_distribution(
    outputs["validation_predictions"],
    paths,
)
print(f"Saved: {probability_distribution_path}")
display(Image(filename=str(probability_distribution_path)))

In [ ]:
error_analysis_path = plot_error_analysis(outputs["error_counts"], paths)
print(f"Saved: {error_analysis_path}")
display(Image(filename=str(error_analysis_path)))

print(
    "False Negative is especially important in employee attrition prediction, "
    "because an employee who may leave but is missed by the model can reduce "
    "the usefulness of later management decisions."
)

In [ ]:
runtime_comparison_path = plot_runtime_comparison(outputs["runtime"], paths)
print(f"Saved: {runtime_comparison_path}")
display(Image(filename=str(runtime_comparison_path)))
display(outputs["runtime"])

In [ ]:
analysis_text = make_analysis_text(outputs["evaluation"])
print(analysis_text)

In [ ]:
test_predictions = predict_test_set(outputs["models"], x_test_full, paths)

print("X_test has no true label, so only predictions and positive probabilities are saved.")
display(test_predictions.head())
print(f"Saved: {RESULTS_DIR / 'test_predictions.csv'}")

## Experiment Summary

In this experiment, four classification models were built for employee attrition prediction: an information gain decision tree, a Gini index decision tree, a Gaussian Naive Bayes model, and an RBF-kernel support vector machine. The available labeled training data from Experiment 2 was split into a training set and a validation set, and the validation set was used for model evaluation. Accuracy, precision, recall, F1-score, ROC-AUC, PR-AUC, confusion matrices, ROC curves, precision-recall curves, feature importance, probability distributions, error analysis, and runtime were compared. Since `X_test_preprocessed.csv` has no true labels, it was only used to generate final predictions. For employee attrition prediction, recall and false-negative analysis are important because missing employees who are likely to leave may weaken later management decisions.